In [34]:
# 必要なライブラリをインポート
import pandas as pd
import joblib # モデルの保存に使用
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import os

# --- 1. データの読み込み ---
features_data_path = '../data/processed/features.parquet'
df = pd.read_parquet(features_data_path)

In [30]:
df.columns

Index(['ID', '種類', '市区町村コード', '都道府県名', '市区町村名', '地区名', '最寄駅：名称', '最寄駅：距離（分）',
       '間取り', '面積（㎡）', '建築年', '建物の構造', '用途', '今後の利用目的', '都市計画', '建ぺい率（％）',
       '容積率（％）', '取引時点', '取引価格（総額）_log', '取引時点での築年数', '人口密度', '市区町村人口密度',
       '取引の事情等_その他事情有り', '取引の事情等_他の権利・負担付き', '取引の事情等_他の権利・負担付き、調停・競売等',
       '取引の事情等_瑕疵有りの可能性', '取引の事情等_調停・競売等', '取引の事情等_調停・競売等、瑕疵有りの可能性',
       '取引の事情等_関係者間取引', '取引の事情等_関係者間取引、瑕疵有りの可能性', '取引の事情等_関係者間取引、調停・競売等',
       '取引の事情等_その他', '改装_改装済', '改装_未改装', '間取り_grouped_その他',
       '間取り_grouped_オープンフロア', '間取り_grouped_欠損値', '間取り_grouped_１ＤＫ',
       '間取り_grouped_１Ｋ', '間取り_grouped_１ＬＤＫ', '間取り_grouped_１Ｒ',
       '間取り_grouped_２ＤＫ', '間取り_grouped_２Ｋ', '間取り_grouped_２ＬＤＫ',
       '間取り_grouped_２ＬＤＫ＋Ｓ', '間取り_grouped_３ＤＫ', '間取り_grouped_３ＬＤＫ',
       '間取り_grouped_４ＤＫ', '間取り_grouped_４ＬＤＫ'],
      dtype='object')

In [35]:
df

,ID,種類,市区町村コード,都道府県名,市区町村名,地区名,最寄駅：名称,最寄駅：距離（分）,間取り,面積（㎡）,...,間取り_grouped_１ＬＤＫ,間取り_grouped_１Ｒ,間取り_grouped_２ＤＫ,間取り_grouped_２Ｋ,間取り_grouped_２ＬＤＫ,間取り_grouped_２ＬＤＫ＋Ｓ,間取り_grouped_３ＤＫ,間取り_grouped_３ＬＤＫ,間取り_grouped_４ＤＫ,間取り_grouped_４ＬＤＫ
0,1060685,中古マンション等,1108,北海道,札幌市厚別区,大谷地東,大谷地,8.0,３ＬＤＫ,80,...,0,0,0,0,0,0,0,1,0,0
1,1005580,中古マンション等,1101,北海道,札幌市中央区,南９条西,中島公園,5.0,１ＤＫ,30,...,0,0,0,0,0,0,0,0,0,0
2,1001363,中古マンション等,1101,北海道,札幌市中央区,北３条西,西１１丁目,11.0,３ＬＤＫ,65,...,0,0,0,0,0,0,0,1,0,0
3,1052374,中古マンション等,1108,北海道,札幌市厚別区,厚別中央２条,ひばりが丘(北海道),5.0,４ＬＤＫ,90,...,0,0,0,0,0,0,0,0,0,1
4,1059107,中古マンション等,1108,北海道,札幌市厚別区,厚別東４条,新さっぽろ,12.0,４ＬＤＫ,80,...,0,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
551640,47001698,中古マンション等,47201,沖縄県,那覇市,松島,市立病院前(沖縄),3.0,４ＬＤＫ,80,...,0,0,0,0,0,0,0,0,0,1
551641,47030340,中古マンション等,47201,沖縄県,那覇市,前島,美栄橋,4.0,２ＤＫ,40,...,0,0,1,0,0,0,0,0,0,0
551642,47000640,中古マンション等,47201,沖縄県,那覇市,字国場,壺川,29.0,３ＬＤＫ,65,...,0,0,0,0,0,0,0,1,0,0
551643,47014718,中古マンション等,47201,沖縄県,那覇市,小禄,奥武山公園,15.0,３ＬＤＫ,65,...,0,0,0,0,0,0,0,1,0,0


In [36]:
from sklearn import linear_model
from sklearn import preprocessing 


### ②説明変数・目的変数のセット ###
# 説明変数のセット
X = df[['最寄駅：距離（分）', '面積（㎡）',  '建ぺい率（％）',
       '容積率（％）','取引時点での築年数', '取引の事情等_その他事情有り',
       '取引の事情等_他の権利・負担付き', '取引の事情等_他の権利・負担付き、調停・競売等', '取引の事情等_瑕疵有りの可能性',
       '取引の事情等_調停・競売等', '取引の事情等_調停・競売等、瑕疵有りの可能性', '取引の事情等_関係者間取引',
       '取引の事情等_関係者間取引、瑕疵有りの可能性', '取引の事情等_関係者間取引、調停・競売等', 
       '改装_改装済',  '間取り_grouped_オープンフロア',
       '間取り_grouped_欠損値', '間取り_grouped_１ＤＫ', '間取り_grouped_１Ｋ',
       '間取り_grouped_１ＬＤＫ', '間取り_grouped_１Ｒ', '間取り_grouped_２ＤＫ',
       '間取り_grouped_２Ｋ', '間取り_grouped_２ＬＤＫ', '間取り_grouped_２ＬＤＫ＋Ｓ',
       '間取り_grouped_３ＤＫ', '間取り_grouped_３ＬＤＫ', '間取り_grouped_４ＤＫ',
       '間取り_grouped_４ＬＤＫ','人口密度','市区町村人口密度']]
# 目的変数のセット
Y = df['取引価格（総額）_log']

### 標準化 ###
# 説明変数の標準化（Zスコア）
X_standard =  preprocessing.scale(X) #各自入力

# 目的変数の標準化（Zスコア）
Y_standard = preprocessing.scale(Y) #各自入力

### ③モデル構築 ###
# 重回帰モデルを作成
model = linear_model.LinearRegression()  # インスタンス化（関数を使える状態にする）
model.fit(X_standard,Y_standard)        #各自入力  # モデル構築（フィッティング）

LinearRegression()

In [37]:
### ④結果の出力 ###

# 各自入力（前出部分からコピペすればOK。ただし、model.score()の中身は、X_standard, Y_standardに変更する必要がある）
df_coef = pd.DataFrame({'Variables':X.columns,
                          'Coefficients':model.coef_ #各自入力 #Pythonでは、カッコ内はいくら改行してもOK
                        })
display( df_coef ) #データフレームを明示的に画面出力したい場合は、display(df)を用いる

# 切片の出力
print( '切片:',  model.intercept_ ) #各自入力

# 決定係数の出力
print( '決定係数:', model.score(X_standard,Y_standard) ) #各自入力

,Variables,Coefficients
0,最寄駅：距離（分）,-0.110086
1,面積（㎡）,0.341621
2,建ぺい率（％）,-0.060798
3,容積率（％）,0.114971
4,取引時点での築年数,-0.433516
5,取引の事情等_その他事情有り,-0.006195
6,取引の事情等_他の権利・負担付き,-0.004597
7,取引の事情等_他の権利・負担付き、調停・競売等,-0.000232
8,取引の事情等_瑕疵有りの可能性,-0.011399
9,取引の事情等_調停・競売等,-0.087589


切片: 5.522293878478562e-15
決定係数: 0.6541025309171189


In [38]:
# --- 5. 学習済みモデルの保存 ---
models_dir = '../models/'
model_file = 'final_model.pkl'
model_path = os.path.join(models_dir, model_file)

# modelsフォルダがなければ作成する
os.makedirs(models_dir, exist_ok=True)

# joblibを使ってモデルを保存
joblib.dump(model, model_path)

print(f"\n学習済みモデルを {model_path} に保存しました。")


学習済みモデルを ../models/final_model.pkl に保存しました。
